# ML-02 Stage 1 Contract Authoring

목적: source parquet의 컬럼 목록, sample 통계, Stage 1 feature 후보를 확인한 뒤 사용자가 직접 최종 `fb_input_feature_contract.csv`를 작성한다.

이 노트북은 feature build를 실행하지 않는다. 최종 contract 저장과 검증까지만 담당한다.

## 실행 흐름

1. source parquet schema와 sample 통계를 확인한다.
2. ML-02 Stage 1 feature 후보를 확인한다.
3. 사용자가 `USER_CONTRACT_ROWS`에 최종 contract row를 직접 작성한다.
4. 최종 contract를 `fb_inputs`에 저장하고 검증한다.

# 00_경로 및 환경설정

In [12]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", 100)

# ============================================================
# 0.1 Runtime Settings
# 실행 환경: ML 담당자 기준
# - Kernel          : Local WSL
# - Code repo       : local Git repo
# - Output artifacts: local Git repo ml/ml-02/fb_inputs/
# ============================================================
SEED = 42

# Root Paths: 다른 환경에서 실행할 경우 보통 이 경로만 수정한다.
LOCAL_REPO_ROOT = Path("/mnt/d/AML_git/Graph_AML").resolve()
DRIVE_REPO_ROOT = Path("/mnt/g/내 드라이브/Graph_AML").resolve()

# Input/Output Paths
ML_02_BASE_DIR = LOCAL_REPO_ROOT / "ml" / "ml-02"
ML_02_FB_INPUTS_DIR = ML_02_BASE_DIR / "fb_inputs"

# Module Code Paths: local Git repo에서 feature_build 모듈을 import한다.
ML_02_MODULE_FB_DIR = ML_02_BASE_DIR / "00_feature_build"

# ============================================================
# 0.2 Path Validation
# ============================================================
def require_dir(path: Path, name: str) -> None:
    path = Path(path).resolve()
    if not path.exists():
        raise FileNotFoundError(f"{name} not found: {path}")
    if not path.is_dir():
        raise NotADirectoryError(f"{name} is not a directory: {path}")

for name, path in {
    "LOCAL_REPO_ROOT": LOCAL_REPO_ROOT,
    "ML_02_BASE_DIR": ML_02_BASE_DIR,
    "ML_02_FB_INPUTS_DIR": ML_02_FB_INPUTS_DIR,
    "ML_02_MODULE_FB_DIR": ML_02_MODULE_FB_DIR,
}.items():
    require_dir(path, name)

# ============================================================
# 0.3 Import Path Setup
# ============================================================
MODULE_DIR = str(ML_02_MODULE_FB_DIR)

if MODULE_DIR not in sys.path:
    sys.path.insert(0, MODULE_DIR)

from ml_02_fb_utils import set_seed
from ml_02_fb_contract import CONTRACT_COLUMNS, CONTRACT_VERSION, validate_fb_input_contract
from ml_02_fb_specs import ml02_stage1_feature_specs

set_seed(SEED)

# 01_입출력 설정

In [13]:
# ============================================================
# 1.1 Run identifiers
# ============================================================
EXPORT_EXPERIMENT_ID = "ml_02"
RUN_ID = "r01"
ARTIFACT_PREFIX = f"{EXPORT_EXPERIMENT_ID}__{RUN_ID}"

# ============================================================
# 1.2 Source inputs
# ============================================================
# ML-02 Stage 1 feature를 만들 원본 parquet 경로다.
SOURCE_PARQUET_PATH = LOCAL_REPO_ROOT / "ml" / "ml-01" / "ml_inputs" / "r02" / "ml_01__r02_Xy_all.parquet"
PARENT_APPROVED_CONTRACT_PATH = LOCAL_REPO_ROOT / "ml" / "ml-01" / "ml_inputs" / "r02" / "ml_01__r02_fb_output_feature_contract_approve.csv"
PARENT_ENCODING_MANIFEST_PATH = LOCAL_REPO_ROOT / "ml" / "ml-01" / "ml_inputs" / "r02" / "ml_01__r02_encoding_manifest.json"

# ============================================================
# 1.3 Output contract path
# ============================================================
FB_INPUT_DIR = ML_02_FB_INPUTS_DIR / RUN_ID
FB_INPUT_CONTRACT_PATH = FB_INPUT_DIR / f"{ARTIFACT_PREFIX}_fb_input_feature_contract.csv"
FB_INPUT_CATEGORY_VALUES_PATH = FB_INPUT_DIR / f"{ARTIFACT_PREFIX}_category_values.json"

# 기존 contract를 덮어쓰려면 True로 바꾼다. 기본은 안전하게 False다.
OVERWRITE_CONTRACT_OUTPUTS = False

PROFILE_ROWS: int | None = None          # 빠른 preview만 필요할 때만 정수를 넣는다. 예: PROFILE_ROWS = 100_000
PROFILE_COLUMNS: list[str] | None = None # None이면 전체 컬럼을 profile한다. 특정 컬럼만 검토하려면 리스트로 지정한다.

STAGE1_FEATURE_SPECS = ml02_stage1_feature_specs(used_in_ml=True)
STAGE1_SPEC_BY_OUTPUT = {spec.output_col: spec for spec in STAGE1_FEATURE_SPECS}

# ============================================================
# Configuration Summary
# ============================================================
print("=" * 80)
print(f"SEED                         : {SEED}")
print("-" * 80)
print(f"EXPORT_EXPERIMENT_ID         : {EXPORT_EXPERIMENT_ID}")
print(f"RUN_ID                       : {RUN_ID}")
print(f"ARTIFACT_PREFIX              : {ARTIFACT_PREFIX}")
print("-" * 80)
print(f"LOCAL_REPO_ROOT              : {LOCAL_REPO_ROOT}")
print(f"DRIVE_REPO_ROOT              : {DRIVE_REPO_ROOT}")
print("-" * 80)
print(f"ML_02_BASE_DIR               : {ML_02_BASE_DIR}")
print(f"SOURCE_PARQUET_PATH          : {SOURCE_PARQUET_PATH}")
print(f"PARENT_APPROVED_CONTRACT_PATH: {PARENT_APPROVED_CONTRACT_PATH}")
print(f"PARENT_ENCODING_MANIFEST_PATH: {PARENT_ENCODING_MANIFEST_PATH}")
print("-" * 80)
print(f"ML_02_FB_INPUTS_DIR          : {ML_02_FB_INPUTS_DIR}")
print(f"FB_INPUT_DIR                 : {FB_INPUT_DIR}")
print(f"FB_INPUT_CONTRACT_PATH       : {FB_INPUT_CONTRACT_PATH}")
print(f"FB_INPUT_CATEGORY_VALUES_PATH: {FB_INPUT_CATEGORY_VALUES_PATH}")
print("-" * 80)
print(f"OVERWRITE_CONTRACT_OUTPUTS   : {OVERWRITE_CONTRACT_OUTPUTS}")
print(f"PROFILE_ROWS                 : {PROFILE_ROWS}")
print(f"PROFILE_COLUMNS              : {PROFILE_COLUMNS}")
print("=" * 80)

SEED                         : 42
--------------------------------------------------------------------------------
EXPORT_EXPERIMENT_ID         : ml_02
RUN_ID                       : r01
ARTIFACT_PREFIX              : ml_02__r01
--------------------------------------------------------------------------------
LOCAL_REPO_ROOT              : /mnt/d/AML_git/Graph_AML
DRIVE_REPO_ROOT              : /mnt/g/내 드라이브/Graph_AML
--------------------------------------------------------------------------------
ML_02_BASE_DIR               : /mnt/d/AML_git/Graph_AML/ml/ml-02
SOURCE_PARQUET_PATH          : /mnt/d/AML_git/Graph_AML/ml/ml-01/ml_inputs/r02/ml_01__r02_Xy_all.parquet
PARENT_APPROVED_CONTRACT_PATH: /mnt/d/AML_git/Graph_AML/ml/ml-01/ml_inputs/r02/ml_01__r02_fb_output_feature_contract_approve.csv
PARENT_ENCODING_MANIFEST_PATH: /mnt/d/AML_git/Graph_AML/ml/ml-01/ml_inputs/r02/ml_01__r02_encoding_manifest.json
--------------------------------------------------------------------------------
ML_02

# 02_source parquet 확인

In [14]:
# ============================================================
# 2.0 Source parquet schema 확인
# ============================================================
# 이 셀은 원본 parquet의 실제 데이터를 전체 로드하지 않고,
# parquet footer에 저장된 metadata와 Arrow schema만 읽는다.
#
# 목적:
# - contract 작성 전에 실제 입력 parquet에 존재하는 컬럼명을 확인한다.
# - source_column / carry_forward 대상 컬럼의 오타를 사전에 줄인다.
# - 예상 dtype과 실제 parquet 저장 dtype이 다른 컬럼을 확인한다.
# - 전체 row count를 metadata 기준으로 확인해 입력 파일이 의도한 파일인지 점검한다.
#
# 주의:
# - 여기서 확인하는 dtype은 pandas dtype이 아니라 parquet의 Arrow dtype이다.
# - 실제 pandas로 로드했을 때의 dtype은 다음 profiling 셀에서 별도로 확인한다.
# - ParquetFile 생성은 metadata/schema 접근용이며, 컬럼 데이터를 메모리에 올리는 작업은 아니다.


# parquet footer metadata와 schema에 접근하기 위한 ParquetFile 객체를 만든다.
source_parquet = pq.ParquetFile(SOURCE_PARQUET_PATH)

# parquet에 저장된 Arrow schema 객체다.
# 각 컬럼의 이름, 저장 타입, nullable 여부 같은 구조 정보를 담고 있다.
source_schema = source_parquet.schema_arrow

# 실제 parquet에 들어 있는 컬럼명 목록이다.
source_columns = list(source_schema.names)

# 컬럼별 Arrow dtype을 dict 형태로 정리한다.
source_arrow_types = {field.name: str(field.type) for field in source_schema}

# parquet metadata에 기록된 전체 row 수다.
# 데이터를 직접 읽지 않고 footer metadata에서 가져오므로 빠르게 확인 가능하다.
source_row_count = int(source_parquet.metadata.num_rows)

# schema 확인용 DataFrame을 만든다.
# column_order: parquet 파일에 저장된 컬럼 순서. 사람이 보기 쉽게 1부터 시작한다.
# column_name : 실제 contract에서 참조 가능한 원본 컬럼명이다.
# arrow_type  : parquet에 저장된 Arrow dtype이다.
source_schema_df = pd.DataFrame(
    [
        {
            "column_order": index,
            "column_name": column,
            "arrow_type": source_arrow_types[column],
        }
        for index, column in enumerate(source_columns, start=1)
    ]
)

# 입력 파일 규모와 컬럼 수를 먼저 요약 출력한다.
# 이 값이 예상과 다르면 이후 contract 작성 전에 SOURCE_PARQUET_PATH부터 재확인한다.
print("source_row_count   :", source_row_count)
print("source_column_count:", len(source_columns))

# 전체 컬럼 목록과 parquet 저장 타입을 표로 확인한다.
# 이 표를 기준으로 USER_CONTRACT_ROWS의 source_column 값을 작성한다.
display(source_schema_df)

source_row_count   : 5077237
source_column_count: 108


,column_order,column_name,arrow_type
0,1,tx_id,string
1,2,timestamp,timestamp[us]
2,3,split,string
3,4,label,int8
4,5,sender_account,string
...,...,...,...
103,104,cat__bank_pair__xgb_cat,"dictionary<values=string, indices=int32, ordered=0>"
104,105,cat__payment_currency__xgb_cat,"dictionary<values=string, indices=int32, ordered=0>"
105,106,cat__receiving_currency__xgb_cat,"dictionary<values=string, indices=int32, ordered=0>"
106,107,cat__currency_pair__xgb_cat,"dictionary<values=string, indices=int32, ordered=0>"


In [15]:
# ============================================================
# 3.0 Source parquet profile 확인
# ============================================================
# 이 셀은 contract 작성에 필요한 컬럼별 기초 통계를 확인한다.
#
# 목적:
# - source parquet의 각 컬럼이 실제로 어떤 값 분포를 가지는지 확인한다.
# - contract에서 feature_type, source_column, dtype, 결측 처리 필요 여부를 판단할 근거를 만든다.
# - 숫자형 컬럼은 min/max/mean으로 값 범위와 이상치를 대략 점검한다.
# - datetime 컬럼은 min/max로 시간 범위를 확인한다.
# - 문자열/object/category 계열 컬럼은 상위 빈도값으로 대표 값을 확인한다.
#
# 주의:
# - PROFILE_ROWS=None이면 지정 컬럼 전체를 pandas DataFrame으로 로드한다.
# - 대용량 parquet에서는 메모리 사용량이 커질 수 있으므로 빠른 확인만 필요하면 PROFILE_ROWS에 정수를 넣는다.
# - PROFILE_ROWS를 사용할 경우 parquet 앞부분 N행만 읽기 때문에 시간순 데이터에서는 초반 구간 편향이 있을 수 있다.
# - 이 profile 결과는 contract 작성 참고용이며, feature build 자체를 실행하지는 않는다.


def read_parquet_profile(
    path: Path,               # profile 대상 parquet 파일 경로.
    columns: list[str],       # 읽을 컬럼 목록. 전체 컬럼 또는 사용자가 지정한 PROFILE_COLUMNS가 들어온다.
    profile_rows: int | None, # None이면 전체 row를 읽고, 양의 정수이면 parquet 앞부분 N행만 읽는다.
) -> tuple[pd.DataFrame, str]:
    """
    parquet 파일에서 profile 계산에 사용할 DataFrame을 읽는다.
        
    Returns
    -------
    tuple[pd.DataFrame, str]
        - profile DataFrame
        - profile mode: "full" 또는 "sample"
    """
    # PROFILE_ROWS=None이면 지정 컬럼의 전체 row를 읽는다.
    if profile_rows is None:
        profile_df = pd.read_parquet(path, columns=columns)
        return profile_df, "full"
    
    # PROFILE_ROWS에 정수를 넣는 경우, 0 이하 값은 의미가 없으므로 즉시 중단한다.
    if profile_rows <= 0:
        raise ValueError("PROFILE_ROWS must be a positive integer or None.")
    
    # 빠른 preview 모드다. parquet 전체를 읽지 않고 iter_batches로 첫 번째 batch만 읽는다.
    parquet_file = pq.ParquetFile(path)
    batches = parquet_file.iter_batches(batch_size=profile_rows, columns=columns)
    try:
        profile_df = next(batches).to_pandas()
        return profile_df, "sample"
    except StopIteration as exc:
        # 파일은 존재하지만 row가 0건이면 통계 계산이 불가능하다.
        raise ValueError(f"parquet file has no rows: {path}") from exc
    
def summarize_profile_column(series: pd.Series) -> dict[str, object]:
    """
    profile DataFrame의 한 컬럼에 대해 contract 작성 참고 통계를 만든다.
    공통 통계:
    - pandas dtype
    - non-null count
    - missing count / missing rate
    - nunique(dropna=True)
    타입별 추가 통계:
    - numeric: min / max / mean
    - datetime: min / max
    - 그 외 문자열/object/category 계열: 상위 5개 빈도값
    """
    # 모든 컬럼에 공통으로 계산하는 품질 지표다.
    # 결측률과 고유값 수는 contract 작성 시 필수 컬럼 여부, categorical 처리 여부,
    result: dict[str, object] = {
        "column_name": series.name,
        "pandas_dtype": str(series.dtype),
        "profile_non_null_count": int(series.notna().sum()),
        "profile_missing_count": int(series.isna().sum()),
        "profile_missing_rate": float(series.isna().mean()),
        "profile_nunique_dropna": int(series.nunique(dropna=True)),
    }
    
    # 숫자형 컬럼은 값 범위와 평균을 확인한다.
    if pd.api.types.is_numeric_dtype(series):
        result.update(
            {
                "profile_min": series.min(skipna=True),
                "profile_max": series.max(skipna=True),
                "profile_mean": series.mean(skipna=True),
            }
        )

    # datetime 컬럼은 시간 범위를 확인한다.
    # split이나 rolling feature 생성에서 시간 범위가 예상과 맞는지 확인하는 데 사용한다.
    elif pd.api.types.is_datetime64_any_dtype(series):
        result.update(
            {
                "profile_min": series.min(skipna=True),
                "profile_max": series.max(skipna=True),
            }
        )

    # 문자열/object/category 계열은 대표값을 확인한다.
    # 은행 코드, 통화, 결제 수단, split 값처럼 범주형 후보 컬럼의 값 분포를 보기 위한 용도다.
    else:
        top_values = series.dropna().astype(str).value_counts().head(5).to_dict()
        result["profile_top_values"] = json.dumps(top_values, ensure_ascii=False)
    return result

# PROFILE_COLUMNS가 None이면 source parquet의 전체 컬럼을 profile한다.
profile_columns = source_columns if PROFILE_COLUMNS is None else [str(column) for column in PROFILE_COLUMNS]

# 사용자가 지정한 PROFILE_COLUMNS 중 source parquet에 없는 컬럼을 먼저 검증한다.
missing_profile_columns = sorted(set(profile_columns) - set(source_columns))
if missing_profile_columns:
    raise ValueError(f"PROFILE_COLUMNS not found in source parquet: {missing_profile_columns[:30]}")

# PROFILE_ROWS 설정에 따라 full 또는 sample profile을 읽는다.
# 반환된 profile_mode는 출력 로그에 남겨 이후 통계가 전체 기준인지 샘플 기준인지 구분한다.
profile_df, profile_mode = read_parquet_profile(
    SOURCE_PARQUET_PATH,
    profile_columns,
    PROFILE_ROWS,
)

# 컬럼별 profile 통계를 계산한다.
# profile_columns 순서를 그대로 사용해 source schema 확인표와 비교하기 쉽게 유지한다.
source_profile_df = pd.DataFrame(
    [summarize_profile_column(profile_df[column]) for column in profile_columns]
)

# profile 실행 범위를 먼저 출력한다. (샘플링 여부 판단)
print("profile_mode        :", profile_mode)
print("profile_rows_config :", PROFILE_ROWS)
print("profile_rows_loaded :", len(profile_df))
print("profile_column_count:", len(profile_columns))

# contract 작성 참고용 컬럼별 통계표를 표시한다.
# 이 표를 보고 USER_CONTRACT_ROWS에 포함할 carry_forward 컬럼과 Stage 1 입력 컬럼을 결정한다.
display(source_profile_df)

profile_mode        : full
profile_rows_config : None
profile_rows_loaded : 5077237
profile_column_count: 108


,column_name,pandas_dtype,profile_non_null_count,profile_missing_count,profile_missing_rate,profile_nunique_dropna,profile_top_values,profile_min,profile_max,profile_mean
0,tx_id,object,5077237,0,0.0,5077237,"{""0"": 1, ""3384823"": 1, ""3384830"": 1, ""3384829"": 1, ""3384828"": 1}",NaN,NaN,NaN
1,timestamp,datetime64[us],5077237,0,0.0,14400,NaN,2022-09-01 00:00:00,2022-09-10 23:59:00,NaN
2,split,string,5077237,0,0.0,3,"{""train"": 3046186, ""val"": 1015633, ""test"": 1015418}",NaN,NaN,NaN
3,label,int8,5077237,0,0.0,2,NaN,0,1,0.000891
4,sender_account,object,5077237,0,0.0,496973,"{""070|100428660"": 168672, ""070|1004286A8"": 103018, ""070|100428978"": 20497, ""070|1004286F0"": 18663, ""070|100428780"": 17264}",NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
103,cat__bank_pair__xgb_cat,category,5077237,0,0.0,262730,"{""__UNKNOWN__"": 23068, ""100187"": 7890, ""37123"": 7172, ""56336"": 7164, ""17573"": 6551}",NaN,NaN,NaN
104,cat__payment_currency__xgb_cat,category,5077237,0,0.0,15,"{""US Dollar"": 1894756, ""Euro"": 1168024, ""Swiss Franc"": 234816, ""Yuan"": 213713, ""Shekel"": 192179}",NaN,NaN,NaN
105,cat__receiving_currency__xgb_cat,category,5077237,0,0.0,15,"{""US Dollar"": 1878952, ""Euro"": 1171743, ""Swiss Franc"": 237819, ""Yuan"": 206520, ""Shekel"": 194982}",NaN,NaN,NaN
106,cat__currency_pair__xgb_cat,category,5077237,0,0.0,209,"{""175"": 1856128, ""55"": 1153530, ""143"": 234393, ""207"": 203506, ""127"": 192063}",NaN,NaN,NaN


# 03_Stage 1 후보 확인

In [16]:
# ============================================================
# 4.0 Stage 1 feature 후보 확인
# ============================================================
# 목적:
# - 사용자가 contract를 직접 작성하기 전에 생성 가능한 Stage 1 feature 후보를 확인한다.
# - 각 feature가 어떤 operation으로 만들어지는지 확인한다.
# - operation별 입력 컬럼과 파라미터를 확인해 contract row 작성 시 참고한다.
# - feature_group과 leakage_risk를 확인해 누수 위험이 있는 feature를 구분한다.
#
# 주의:
# - 이 셀은 후보 목록만 표로 보여준다.
# - 실제 feature parquet를 생성하지 않는다.
# - 최종 생성 여부는 이후 USER_CONTRACT_ROWS에 어떤 feature를 넣는지에 따라 결정된다.
# - 아래 column_name 값은 stage1_contract_row("...")에 넣을 수 있는 후보 이름이다.

# Stage 1 feature spec 목록을 사람이 검토하기 쉬운 DataFrame으로 변환한다.
# STAGE1_FEATURE_SPECS는 ml02_stage1_feature_specs(used_in_ml=True)에서 생성된 spec 객체 목록이다.
stage1_candidates_df = pd.DataFrame(
    [
        {
            # 최종 feature build 결과에 생성될 feature 컬럼명이다.
            # contract 작성 시 stage1_contract_row("column_name") 형태로 참조한다.
            "column_name": spec.output_col,

            # feature를 생성하는 연산 이름이다.
            # 예: rolling count, rolling sum, time delta 등 Stage 1 정의에 따른 operation이 들어간다.
            "operation": spec.operation,

            # operation에 필요한 입력 컬럼 역할과 실제 source column 이름이다.
            # dict를 JSON 문자열로 바꿔 표에서 한눈에 확인하기 쉽게 만든다.
            "input_cols": json.dumps(dict(spec.input_cols), ensure_ascii=False, sort_keys=True),

            # window, aggregation 방식 등 operation별 파라미터다.
            # JSON 문자열로 표시해 contract 작성 시 그대로 확인할 수 있게 한다.
            "params": json.dumps(dict(spec.params), ensure_ascii=False, sort_keys=True),

            # feature 계열 또는 family 정보다.
            # 후보를 금액, 시간, 계좌, 거래 관계 등 그룹 단위로 묶어 검토할 때 사용한다.
            "feature_group": spec.family,

            # 시간 누수 또는 split contamination 관련 정책/위험 설명이다.
            # contract에 포함하기 전에 이 값을 보고 누수 위험을 확인한다.
            "leakage_risk": spec.leakage_policy,

            # spec 정의상 기본적으로 ML 입력에 사용할지 여부다.
            # 최종 used_in_ml 값은 contract 작성 단계에서 다시 확정한다.
            "used_in_ml_default": spec.used_in_ml,
        }
        for spec in STAGE1_FEATURE_SPECS
    ]
)

# 현재 spec 기준으로 확인 가능한 Stage 1 feature 후보 개수를 출력한다.
print("stage1_candidate_count:", len(stage1_candidates_df))

# contract 작성 시 아래 column_name 값을 stage1_contract_row("...")에 넣는다.
display(stage1_candidates_df)

stage1_candidate_count: 28


,column_name,operation,input_cols,params,feature_group,leakage_risk,used_in_ml_default
0,accountstats__sender__out__amount__mean__w1d,rolling_agg,"{""entity_col"": ""sender_account_id"", ""timestamp_col"": ""timestamp"", ""value_col"": ""amount""}","{""agg"": ""mean"", ""closed"": ""left"", ""fill_value"": 0.0, ""window"": ""1d""}",accountstats,past-only; rolling window uses closed=left,True
1,accountstats__receiver__in__amount__mean__w1d,rolling_agg,"{""entity_col"": ""receiver_account_id"", ""timestamp_col"": ""timestamp"", ""value_col"": ""amount_received""}","{""agg"": ""mean"", ""closed"": ""left"", ""fill_value"": 0.0, ""window"": ""1d""}",accountstats,past-only; rolling window uses closed=left,True
2,accountstats__sender__out__amount__std__w1d,rolling_agg,"{""entity_col"": ""sender_account_id"", ""timestamp_col"": ""timestamp"", ""value_col"": ""amount""}","{""agg"": ""std"", ""closed"": ""left"", ""fill_value"": 0.0, ""window"": ""1d""}",accountstats,past-only; rolling window uses closed=left,True
3,accountstats__receiver__in__amount__std__w1d,rolling_agg,"{""entity_col"": ""receiver_account_id"", ""timestamp_col"": ""timestamp"", ""value_col"": ""amount_received""}","{""agg"": ""std"", ""closed"": ""left"", ""fill_value"": 0.0, ""window"": ""1d""}",accountstats,past-only; rolling window uses closed=left,True
4,accountstats__sender__out__amount__mean__w3d,rolling_agg,"{""entity_col"": ""sender_account_id"", ""timestamp_col"": ""timestamp"", ""value_col"": ""amount""}","{""agg"": ""mean"", ""closed"": ""left"", ""fill_value"": 0.0, ""window"": ""3d""}",accountstats,past-only; rolling window uses closed=left,True
5,accountstats__receiver__in__amount__mean__w3d,rolling_agg,"{""entity_col"": ""receiver_account_id"", ""timestamp_col"": ""timestamp"", ""value_col"": ""amount_received""}","{""agg"": ""mean"", ""closed"": ""left"", ""fill_value"": 0.0, ""window"": ""3d""}",accountstats,past-only; rolling window uses closed=left,True
6,accountstats__sender__out__amount__std__w3d,rolling_agg,"{""entity_col"": ""sender_account_id"", ""timestamp_col"": ""timestamp"", ""value_col"": ""amount""}","{""agg"": ""std"", ""closed"": ""left"", ""fill_value"": 0.0, ""window"": ""3d""}",accountstats,past-only; rolling window uses closed=left,True
7,accountstats__receiver__in__amount__std__w3d,rolling_agg,"{""entity_col"": ""receiver_account_id"", ""timestamp_col"": ""timestamp"", ""value_col"": ""amount_received""}","{""agg"": ""std"", ""closed"": ""left"", ""fill_value"": 0.0, ""window"": ""3d""}",accountstats,past-only; rolling window uses closed=left,True
8,accountstats__sender__out__amount__mean__w7d,rolling_agg,"{""entity_col"": ""sender_account_id"", ""timestamp_col"": ""timestamp"", ""value_col"": ""amount""}","{""agg"": ""mean"", ""closed"": ""left"", ""fill_value"": 0.0, ""window"": ""7d""}",accountstats,past-only; rolling window uses closed=left,True
9,accountstats__receiver__in__amount__mean__w7d,rolling_agg,"{""entity_col"": ""receiver_account_id"", ""timestamp_col"": ""timestamp"", ""value_col"": ""amount_received""}","{""agg"": ""mean"", ""closed"": ""left"", ""fill_value"": 0.0, ""window"": ""7d""}",accountstats,past-only; rolling window uses closed=left,True


# 04_사용자 contract 작성

In [17]:
# ============================================================
# 4.0 사용자 contract 작성
# ============================================================
# 이 셀에서 최종 feature contract row를 사용자가 직접 작성한다.
#
# 목적:
# - source parquet에 이미 존재하는 컬럼을 contract에 보존하거나 ML 입력으로 선택한다.
# - Stage 1 feature builder가 새로 만들 feature를 contract에 등록한다.
# - categorical feature를 XGBoost native categorical 방식으로 사용할지 명시한다.
# - label, split, tx_id 같은 metadata/target 계열 컬럼이 ML 입력에 섞이지 않도록 row 단위로 관리한다.
#
# contract에서 인코딩을 지정하는 핵심 컬럼:
# - encoding
# - xgb_feature_type
#
# 권장 기준:
# - 연속형 수치 feature        : encoding="passthrough", xgb_feature_type="q"
# - XGBoost native categorical : encoding="xgb_native", xgb_feature_type="c"
#
# 주의:
# - sender_account_id, receiver_account_id 같은 ID성 고카디널리티 컬럼은 categorical로 넣지 않는다.
# - used_in_ml은 문자열 "TRUE"/"FALSE" 기준으로 기록된다. bool True/False와 혼용하지 않는 것이 안전하다.
# - 이 셀은 contract row를 만드는 helper 정의 단계이며, 실제 feature build나 모델 학습은 수행하지 않는다.


# IBM 원본 거래 CSV에서 온 raw 컬럼명 목록이다.
# 이 목록에 포함되면 column_origin="raw_source"로 분류하고,
# 그 외 컬럼은 전처리 과정에서 생성된 preprocessing 컬럼으로 분류한다.
RAW_SOURCE_COLUMNS = {
    "Timestamp",
    "From Bank",
    "Account",
    "To Bank",
    "Account.1",
    "Amount Received",
    "Receiving Currency",
    "Amount Paid",
    "Payment Currency",
    "Payment Format",
    "Is Laundering",
}


# 앞뒤 공백을 제거한 컬럼명이 RAW_SOURCE_COLUMNS에 있으면 원본 컬럼으로 본다.
# 그 외에는 Part 1 또는 ML-00 전처리에서 생성된 컬럼으로 본다.
def infer_source_column_origin(column_name: str) -> str:
    """source parquet column이 raw 원본인지 전처리 산출물인지 구분한다."""
    return "raw_source" if str(column_name).strip() in RAW_SOURCE_COLUMNS else "preprocessing"


# CONTRACT_COLUMNS에 정의된 contract schema를 기준으로 모든 필드를 빈 문자열로 초기화한다.
# 이후 각 helper가 필요한 필드만 row.update()로 덮어쓴다.
def _base_contract_row() -> dict[str, object]:
    """feature contract의 전체 컬럼을 가진 빈 row를 만든다."""
    return {column: "" for column in CONTRACT_COLUMNS}


def carry_forward_contract_row(
    column_name: str,
    *,
    used_in_ml: str = "TRUE",
    encoding: str = "passthrough",
    xgb_feature_type: str = "q",
    feature_group: str = "source",
    leakage_risk: str = "reviewed",
    selection_note: str = "user selected source parquet column",
) -> dict[str, object]:
    """
    source parquet에 이미 존재하는 컬럼을 그대로 ML 입력에 포함한다.
    사용 예:
    - 금액, 비율 같은 수치형 source feature
    - 이미 전처리된 current-row feature
    기본값:
    - encoding="passthrough"
    - xgb_feature_type="q"
    즉, 별도 인코딩 없이 수치형 feature로 사용한다.
    """
    # source parquet에 이미 존재하는 컬럼을 그대로 ML 입력 후보에 넣는 helper다.
    # 기본값은 수치형 feature 기준이다.
    # categorical로 쓰고 싶으면 categorical_contract_row()를 사용한다.
    # 사용자가 입력한 컬럼명이 실제 source parquet에 없으면 즉시 중단한다.
    if column_name not in source_columns:
        raise ValueError(f"carry_forward source column not found: {column_name}")
    
    # 고정 schema를 가진 빈 contract row를 만든다.
    row = _base_contract_row()

    # source parquet에 이미 materialized된 컬럼을 carry_forward로 등록한다.
    row.update(
        {           
            "contract_version": CONTRACT_VERSION,   # contract schema 버전이다.            
            "artifact_prefix": ARTIFACT_PREFIX,     # 이번 run의 파일 prefix다.            
            "column_name": column_name,             # 최종 ML 입력에서 사용할 feature 컬럼명이다.

            # TRUE이면 학습 feature 후보로 사용한다.
            # FALSE이면 contract에는 남기지만 ML 입력에서는 제외한다.
            "used_in_ml": used_in_ml,

            "source_column": column_name,           # carry_forward에서는 source 컬럼명과 output 컬럼명이 같다.
            "column_origin": infer_source_column_origin(column_name),
            "encoding": encoding,                   # passthrough: 별도 인코딩 없이 그대로 사용
            "build_action": "carry_forward",        # 이미 source parquet에 있는 컬럼이므로 새로 build하지 않는다.            
            "build_in_fb": "FALSE",                 # feature_build 단계에서 계산하지 않는다는 뜻이다.
            "xgb_feature_type": xgb_feature_type,   # q: numeric/quantitative, c: categorical            
            "feature_group": feature_group,         # feature 계열 구분용 메타데이터다.            
            "leakage_risk": leakage_risk,           # 누수 검토 결과 또는 주의사항을 기록한다.            
            "review_status": "approved",            # 사용자가 승인한 row임을 표시한다.            
            "selection_note": selection_note,       # 왜 선택했는지 사람이 읽을 수 있는 메모다.            
            "contract_row_origin": "user_authored", # 자동 생성이 아니라 사용자가 직접 작성한 row임을 표시한다.            
            "parent_artifact_prefix": "",           # 현재 run에서 직접 작성한 row이므로 비워둔다.            
            "source_contract_path": "",             # source contract path를 쓰지 않으므로 비워둔다.            
            "source_data_path": str(SOURCE_PARQUET_PATH), # 이 contract가 참조한 source parquet 경로다.            
            "materialized": "TRUE",                 # source parquet에 이미 존재하는 컬럼이라는 뜻이다.            
            "note": "source parquet column",        # 사람이 읽는 보조 메모다.
        }
    )
    return row


def categorical_contract_row(
    column_name: str,
    *,
    used_in_ml: str = "TRUE",
    feature_group: str = "categorical",
    leakage_risk: str = "reviewed_low_cardinality",
    selection_note: str = "user selected low-cardinality categorical source column",
) -> dict[str, object]:
    """
    source parquet에 이미 존재하는 low-cardinality 컬럼을  XGBoost native categorical feature로 포함한다.
    사용 예:
    - time__row__dayofweek, time__row__is_weekend
    - bank / currency / payment_format 계열
      단, 실제 column_name은 source_schema_df에서 확인한 이름을 사용한다.
    이 helper는 아래 값을 고정한다.
    - encoding="xgb_native"
    - xgb_feature_type="c"
    주의:
    - cardinality가 높은 ID 컬럼에는 사용하지 않는다.
    - 예: sender_account_id, receiver_account_id는 제외 권장.
    """
    # source parquet에 이미 존재하는 low-cardinality 컬럼을 categorical feature로 등록한다.
    # 예: 요일, 주말 여부, 통화, 결제수단 등
    # carry_forward_contract_row를 재사용하되 categorical 설정만 고정한다.
    return carry_forward_contract_row(
        column_name,
        used_in_ml=used_in_ml,        
        encoding="xgb_native",        # XGBoost native categorical 인코딩을 사용한다.        
        xgb_feature_type="c",         # XGBoost feature type을 categorical로 지정한다.        
        feature_group=feature_group,  # 사용자가 지정한 feature group을 그대로 기록한다.        
        leakage_risk=leakage_risk,    # low-cardinality 검토가 끝난 categorical이라는 의미의 메타데이터다.        
        selection_note=selection_note,# 선택 사유 메모다.
    )


def source_inventory_contract_row(
    column_name: str,
    *,
    used_in_ml: str = "FALSE",
    encoding: str = "passthrough",
    xgb_feature_type: str = "",
    selection_note: str = "source parquet column preserved in contract and parquet",
) -> dict[str, object]:
    """
    source parquet의 전체 컬럼을 contract에 보존한다.
    기본값은 used_in_ml=FALSE다. 즉, parquet와 contract에는 남기되 모델 입력 feature로는 쓰지 않는다.
    선택 source feature는 categorical_contract_row() 또는 carry_forward_contract_row()로 override한다.
    """
    # 입력 검증: 이 함수는 source parquet에 실제 존재하는 컬럼만 contract inventory로 등록한다.
    if column_name not in source_columns:
        raise ValueError(f"source inventory column not found: {column_name}")
    
    # 컬럼명 정규화: contract에 기록될 column_name/source_column 값을 통일하기 위해 문자열화 후 앞뒤 공백을 제거한다.
    normalized = str(column_name).strip()

    # 라벨/타깃 후보 탐지를 위해 소문자 버전을 별도로 만든다. 원본 표기(normalized)는 contract의 실제 컬럼명으로 유지한다.
    normalized_lower = normalized.lower()

    # 컬럼 분류 룰:
    # tx_id, timestamp, split은  metadata로 분류한다.
    if normalized in {"tx_id", "timestamp", "split"}:
        feature_group = "metadata"
        leakage_risk = "metadata_not_model_feature"

    # 라벨/타깃 계열 컬럼 분류: label, target, y 또는 laundering이 포함된 컬럼은 정답 정보 또는 정답 유사 정보일 가능성이 높다.
    elif normalized == "label" or normalized_lower in {"is laundering", "is_laundering", "target", "y"} or "laundering" in normalized_lower:
        feature_group = "target_metadata"
        leakage_risk = "target_like_source_column_not_for_ml"

    # 일반 source 컬럼: 위 metadata/target 룰에 걸리지 않은 source parquet 컬럼은 inventory로 보존한다.
    else:
        feature_group = "source_parquet_inventory"
        leakage_risk = "inventory_not_selected"

    # contract row 기본값 생성: _base_contract_row()는 contract에 필요한 공통 필드와 기본값을 반환하는 함수이다. 
    # 이후 row.update()에서 이 source 컬럼에 맞는 값만 덮어쓴다.
    row = _base_contract_row()

    # contract metadata 업데이트: 이 블록이 실제 결과물 관리의 핵심이다.
    # 각 source 컬럼이 어떤 artifact/version/path에서 왔고,
    # 모델에 쓰이는지, 어떤 dtype인지, leakage 위험은 무엇인지 contract에 기록한다.
    row.update(
        {
            
            "contract_version": CONTRACT_VERSION,                    # CONTRACT_VERSION 변경 시 같은 컬럼이라도 contract 해석 기준이 달라질 수 있다.
            "artifact_prefix": ARTIFACT_PREFIX,                      # 이 row가 어떤 artifact 묶음에 속하는지 식별하는 prefix.
            "column_name": normalized,                               # 정규화된 이름을 사용해 앞뒤 공백으로 인한 불일치를 줄인다.
            "used_in_ml": used_in_ml,                                # 모델 입력 사용 여부.
            "source_column": normalized,                             # 원천 parquet에서 온 컬럼명.
            "column_origin": infer_source_column_origin(normalized), # 원천 컬럼의 출처 분류.

            "encoding": encoding,               # 인코딩 방식.
            "build_action": "carry_forward",    # feature build 단계에서 수행할 동작.
            "build_in_fb": "FALSE",             # FB(feature builder 또는 feature block으로 추정)에서 생성되는 컬럼인지 여부.

            # XGBoost feature type.
            # 기본값은 빈 문자열이므로 모델 feature로 쓰지 않는 inventory 컬럼에는 타입을 비워둔다.
            # used_in_ml=TRUE로 override할 경우 이 값도 함께 검토해야 한다.
            "xgb_feature_type": xgb_feature_type,

            # 앞에서 분류한 feature 그룹.
            # metadata, target_metadata, source_parquet_inventory 중 하나가 들어간다.
            "feature_group": feature_group,
            
            
            "dtype": source_arrow_types.get(normalized, ""), # source parquet에서 관측된 Arrow dtype.
            "leakage_risk": leakage_risk,  # leakage 위험 분류.
            # 리뷰 상태. inventory는 아직 ML feature로 선택되지 않은 원천 컬럼 목록이라는 의미로 보인다.
            "review_status": "inventory",

            # 모델에서 제외된 이유.
            # used_in_ml이 정확히 문자열 "FALSE"일 때만 not_selected_for_ml을 기록한다.
            # "false", False, "False"를 넣으면 빈 문자열이 되므로 호출부 입력값을 주의해야 한다.
            "excluded_reason": "not_selected_for_ml" if used_in_ml == "FALSE" else "",

            # 선택 또는 제외 사유를 사람이 읽을 수 있게 남기는 설명 필드.
            "selection_note": selection_note,

            # 이 contract row가 source inventory 자동 등록에서 왔음을 표시한다.
            # 다른 row 생성 함수에서 만든 row와 구분하는 추적 필드다.
            "contract_row_origin": "source_inventory",

            # 이 컬럼이 나온 source parquet 경로.
            # 실제 파일 존재 여부나 버전 고정 여부는 이 함수에서 검증하지 않는다. 확인 필요.
            "source_data_path": str(SOURCE_PARQUET_PATH),
            
            # parquet/contract에 실제 materialized된 컬럼으로 취급한다.
            # 단, 모델 입력 사용 여부와는 별개다.
            "materialized": "TRUE",

            # 관측 dtype을 별도 필드에도 기록한다.
            # dtype과 같은 값을 넣고 있어 downstream에서 기대하는 스키마 호환 목적일 수 있다. 확인 필요.
            "observed_dtype": source_arrow_types.get(normalized, ""),

            # 사람이 읽는 보조 설명.
            # 기본 정책이 "보존하지만 ML에는 기본 미사용"임을 명시한다.
            "note": "source parquet column preserved; not selected for ML by default",
        }
    )
    # 최종 출력:
    # 호출부가 반환된 row들을 모아 contract CSV/JSON/Parquet 등으로 저장할 가능성이 있다. 확인 필요.
    return row


def encode_contract_row(
    output_col: str,
    source_column: str,
    *,
    used_in_ml: str = "TRUE",
    encoding: str = "xgb_native",
    xgb_feature_type: str = "c",
    feature_group: str = "categorical_xgb_native",
    leakage_risk: str = "low_if_train_only_category_manifest",
    selection_note: str = "ML-00 native categorical feature inherited for ML-02",
) -> dict[str, object]:
    """source column을 현재 ML-02 build 단계에서 native categorical feature로 encode한다."""

    # 입력 source 컬럼 검증: source_column은 원천 parquet에 이미 존재하는 컬럼이어야 한다.
    if source_column not in source_columns:
        raise ValueError(f"encode source column not found: {source_column}")
    
    # output 컬럼명 충돌 방지: output_col은 현재 build 단계에서 새로 만들어질 categorical feature 이름이다.
    if output_col in source_columns:
        raise ValueError(f"encode output column already exists in source parquet: {output_col}")
    
    # contract row 기본값 생성:
    # 아래 row.update()에서 현재 categorical encoding feature에 필요한 필드만 덮어쓴다.
    row = _base_contract_row()

    # categorical encoding feature에 대한 contract metadata를 기록한다.
    # 이 함수는 실제 feature 값을 생성하지 않고, 이후 build 단계가 따라야 할 "설계서"를 만든다.
    row.update(
        {
            "contract_version": CONTRACT_VERSION,         # contract 산출물의 버전.
            "artifact_prefix": ARTIFACT_PREFIX,           # artifact 묶음을 식별하는 prefix.
            "column_name": output_col,                    # 최종 feature 컬럼명.
            "used_in_ml": used_in_ml,                     # 모델 입력 사용 여부.
            "source_column": source_column,               # 이 feature가 참조하는 원천 parquet 컬럼명.            
            "column_origin": "current_build",             # 컬럼 생성 출처.
            "encoding": encoding,                         # 인코딩 방식.            
            "build_action": "encode",                     # feature build 단계에서 수행할 동작.            
            "build_in_fb": "TRUE",                        # feature builder 단계에서 생성해야 하는 컬럼임을 표시한다.            
            "xgb_feature_type": xgb_feature_type,         # XGBoost feature type.            
            "feature_group": feature_group,               # feature 그룹.
            "dtype": "category",                          # materialized feature의 기대 dtype.            
            "leakage_risk": leakage_risk,                 # leakage 위험 설명.
            "review_status": "approved",                  # 리뷰 상태.
            "selection_note": selection_note,             # feature 선택 근거.
            "contract_row_origin": "user_authored",       # row 작성 출처.
            "source_data_path": str(SOURCE_PARQUET_PATH), # 이 feature가 참조하는 source parquet 경로.     
            "feature_spec_name": encoding,                # feature spec 이름.

            # encoding 상세 파라미터.
            # JSON 문자열로 저장하므로 contract CSV/Parquet에서도 구조화된 설정을 보존할 수 있다.
            # fit_split="train"은 category mapping/manifest를 train 기준으로만 학습하겠다는 의도로 보인다.
            # validation/test 전체를 보고 category를 fit하면 temporal leakage 또는 split leakage가 생길 수 있으므로 구현 확인이 필요하다.
            "encoding_params": json.dumps(
                {"source_column": source_column, "encoding": encoding, "fit_split": "train"},
                ensure_ascii=False,
                sort_keys=True,
            ),

            # 이 시점의 materialization 여부.
            # FALSE이므로 contract에는 정의됐지만 실제 output_col이 feature parquet에 아직 생성되지 않은 상태다.
            "materialized": "FALSE",
            
            # 후속 처리 위치에 대한 메모.
            # 실제 인코딩은 01_ml_02_fb_build_features.ipynb에서 Stage 1 build 이후 수행되는 것으로 기록되어 있다.
            # 해당 notebook이 이 contract row를 읽어 output_col을 생성하는지 확인 필요.
            "note": "to be encoded after Stage 1 build by 01_ml_02_fb_build_features.ipynb",
        }
    )
    # 최종 출력:
    # 파일 저장, feature 생성, 모델 학습, 평가는 여기서 수행하지 않는다.
    # 호출부가 반환된 row를 contract 테이블에 모아 저장하고, 이후 build 단계가 이를 읽어 feature를 생성할 가능성이 높다.
    return row


def stage1_contract_row(
    output_col: str,
    *,
    used_in_ml: str = "TRUE",
    selection_note: str = "user selected Stage 1 time-history feature",
) -> dict[str, object]:
    """
    Stage 1 feature builder가 새로 만들 time-history feature를 contract에 추가한다.
    Stage 1 feature의 성격:
    - rolling count
    - rolling sum / mean / std / max
    - current amount vs rolling mean ratio
    - recency seconds
    - cumulative count
    이들은 대부분 연속형 수치 feature이므로 아래 값을 고정한다.
    - encoding="passthrough"
    - xgb_feature_type="q"
    """
    # Stage 1 feature builder가 새로 계산할 feature를 contract에 등록한다.
    # rolling count/sum/mean/std/max, ratio, recency, cumulative count 등이 대상이다.
    # 이 feature들은 기본적으로 수치형이므로 encoding="passthrough", xgb_feature_type="q"로 고정한다.
    # 사용자가 입력한 output_col이 Stage 1 후보 목록에 없으면 즉시 중단한다.
    if output_col not in STAGE1_SPEC_BY_OUTPUT:
        raise ValueError(f"unknown Stage 1 feature output_col: {output_col}")
    
    # output_col에 해당하는 FeatureSpec을 가져온다.
    # FeatureSpec에는 operation, input_cols, params, family, leakage_policy가 들어 있다.
    spec = STAGE1_SPEC_BY_OUTPUT[output_col]

    # 고정 schema를 가진 빈 contract row를 만든다.
    row = _base_contract_row()
    
    # feature_build에서 새로 계산할 Stage 1 feature row를 등록한다.
    row.update(
        {
            "contract_version": CONTRACT_VERSION, # contract schema 버전이다.
            "artifact_prefix": ARTIFACT_PREFIX,   # 이번 run의 파일 prefix다.            
            "column_name": spec.output_col,       # feature build 후 생성될 최종 feature 컬럼명이다.

            # TRUE이면 학습 feature 후보로 사용한다.
            # FALSE이면 build는 가능하지만 ML 입력에서는 제외한다.
            "used_in_ml": used_in_ml,
            
            "source_column": spec.output_col,      # build feature는 생성 후 같은 이름으로 저장되므로 output_col을 source_column에도 둔다.            
            "column_origin": "current_build",
            "encoding": "passthrough",             # Stage 1 feature는 수치형 계산 결과이므로 별도 인코딩 없이 사용한다.            
            "build_action": "build",               # feature_build 단계에서 새로 계산한다는 뜻이다.            
            "build_in_fb": "TRUE",                 # feature_build 실행 대상임을 표시한다.            
            "xgb_feature_type": "q",               # Stage 1 time-history feature는 수치형으로 사용한다.            
            "feature_group": spec.family,          # FeatureSpec에 정의된 feature family다.      
            "leakage_risk": spec.leakage_policy,   # FeatureSpec에 정의된 누수 정책이다. 예: past-only, closed=left 등            
            "review_status": "approved",           # 사용자가 승인한 row임을 표시한다.            
            "selection_note": selection_note,      # 왜 선택했는지 사람이 읽을 수 있는 메모다.            
            "contract_row_origin": "user_authored",# 자동 생성이 아니라 사용자가 직접 작성한 row임을 표시한다.            
            "parent_artifact_prefix": "",          # 현재 run에서 직접 작성한 row이므로 비워둔다.            
            "source_contract_path": "",            # source contract path를 쓰지 않으므로 비워둔다.            
            "source_data_path": str(SOURCE_PARQUET_PATH), # 이 contract가 참조한 source parquet 경로다.            
            "feature_spec_name": spec.operation,          # 어떤 operation으로 생성할 feature인지 기록한다.

            # feature 생성에 필요한 입력 컬럼과 operation 파라미터를 JSON 문자열로 저장한다.
            # 재현성과 검토를 위한 metadata다.
            "encoding_params": json.dumps(
                {
                    "input_cols": dict(spec.input_cols),
                    "params": dict(spec.params),
                },
                ensure_ascii=False,
                sort_keys=True,
                default=str,
            ),
            
            "materialized": "FALSE",   # 아직 source parquet에 존재하지 않고 feature_build에서 생성될 컬럼이다.
            "note": "to be built by 01_ml_02_fb_build_features.ipynb", # 사람이 읽는 보조 메모다.
        } 
    )
    return row

In [18]:
# ============================================================
# 4.1 최종 contract row 선택
# ============================================================
# 이 셀이 USER_CONTRACT_ROWS를 만들며, 이후 단계(write/validate)는 이 리스트를 단일 source of truth로 사용한다.
#
# 구성 원칙
# - source parquet의 모든 컬럼은 inventory row로 contract에 빠짐없이 남긴다.(parquet 스키마와 contract row 집합의 정합성 확인용)
# - inventory row 기본값은 used_in_ml=FALSE → contract에는 남지만 ML 입력에서는 제외.
# - ML-01 승인 contract의 used_in_ml=TRUE feature를 SOURCE_CONTRACT_OVERRIDES로 carry_forward한다.
# - ML-01 승인본에 이미 materialized된 categorical feature는 중복 encode하지 않는다.
# - 이번 build 단계에서 새로 만드는 row는 ML-02 Stage 1 accountstats 수치형 후보 28개다.
#
# helper 호출 시 의미
# - carry_forward_contract_row(col): source parquet에 이미 있는 컬럼을 그대로 ML 입력으로 사용
#     · 기본 encoding=passthrough, xgb_feature_type="q"(수치형), build_in_fb=FALSE
# - encode_contract_row(out, src): build 단계에서 src 컬럼을 받아 out 컬럼으로 새로 생성(현재 ML-02 r00에서는 사용하지 않음)
# - source_inventory_contract_row(col): used_in_ml=FALSE로 inventory만 남김
# - stage1_contract_row(name): Stage 1 spec 기반 신규 수치형 feature row

SOURCE_CONTRACT_OVERRIDES = {
    # --- ML-00 current-row amount features ---
    # 거래 1건의 송금/수취 금액 및 그 log1p, 송금/수취 비율.
    # 이미 source parquet에 materialized된 수치형이므로 carry_forward로 그대로 사용한다.
    "amount__current__value":          carry_forward_contract_row("amount__current__value",          feature_group="amount"),
    "amount__current__log1p":          carry_forward_contract_row("amount__current__log1p",          feature_group="amount"),
    "amount_received__current__value": carry_forward_contract_row("amount_received__current__value", feature_group="amount"),
    "amount_received__current__log1p": carry_forward_contract_row("amount_received__current__log1p", feature_group="amount"),
    "amount__paid_recv_ratio":         carry_forward_contract_row("amount__paid_recv_ratio",         feature_group="amount"),

    # --- Current-row time source columns ---
    # 원본 time 컬럼(hour/dayofweek/is_weekend)은 contract에는 남기되 ML 입력 feature로는 사용하지 않는다.
    # 실제 ML에 들어가는 것은 아래 TIME_NATIVE_CATEGORICAL_ROWS에서 만든 xgb_native categorical 인코딩 row다.
    # → 같은 의미의 컬럼이 수치형/카테고리형 두 벌로 들어가는 중복을 막기 위한 override.
    "time__row__dayofweek": carry_forward_contract_row(
        "time__row__dayofweek",
        used_in_ml="FALSE",          # ML 입력 제외 (categorical 인코딩 row가 대신 사용됨)
        xgb_feature_type="",         # 미사용이므로 feature type 비움
        feature_group="time",
        selection_note="source time column retained; xgb_native encoded row selected for ML",
    ),
    "time__row__hour": carry_forward_contract_row(
        "time__row__hour",
        used_in_ml="FALSE",
        xgb_feature_type="",
        feature_group="time",
        selection_note="source time column retained; xgb_native encoded row selected for ML",
    ),
    "time__row__is_weekend": carry_forward_contract_row(
        "time__row__is_weekend",
        used_in_ml="FALSE",
        xgb_feature_type="",
        feature_group="time",
        selection_note="source time column retained; xgb_native encoded row selected for ML",
    ),
}

# ML-02 입력은 ML-01 승인본이므로 ML-01에서 이미 선택된 feature는 새로 encode/build하지 않고 carry_forward한다.
# parent approve contract의 used_in_ml=TRUE row가 ML-01 승인 feature의 단일 source of truth다.
if not PARENT_APPROVED_CONTRACT_PATH.is_file():
    raise FileNotFoundError(f"parent approved contract not found: {PARENT_APPROVED_CONTRACT_PATH}")
if not PARENT_ENCODING_MANIFEST_PATH.is_file():
    raise FileNotFoundError(f"parent encoding manifest not found: {PARENT_ENCODING_MANIFEST_PATH}")

parent_contract_df = pd.read_csv(PARENT_APPROVED_CONTRACT_PATH, encoding="utf-8-sig", dtype="string")
parent_encoding_manifest = json.loads(PARENT_ENCODING_MANIFEST_PATH.read_text(encoding="utf-8"))
parent_required_columns = {"column_name", "used_in_ml", "xgb_feature_type"}
parent_missing_columns = parent_required_columns - set(parent_contract_df.columns)
if parent_missing_columns:
    raise ValueError(f"parent approved contract is missing columns: {sorted(parent_missing_columns)}")

parent_selected_df = parent_contract_df[
    parent_contract_df["used_in_ml"].astype(str).str.strip() == "TRUE"
].copy()
parent_selected_df["column_name"] = parent_selected_df["column_name"].astype(str).str.strip()
parent_missing_source_columns = sorted(set(parent_selected_df["column_name"]) - set(source_columns))
if parent_missing_source_columns:
    raise ValueError(
        "ML-01 approved feature columns are missing from ML-02 source parquet. "
        f"missing={parent_missing_source_columns[:30]}, missing_count={len(parent_missing_source_columns)}"
    )

for _, parent_row in parent_selected_df.iterrows():
    column = str(parent_row["column_name"]).strip()
    xgb_feature_type = str(parent_row.get("xgb_feature_type", "")).strip() or "q"
    SOURCE_CONTRACT_OVERRIDES[column] = carry_forward_contract_row(
        column,
        xgb_feature_type=xgb_feature_type,
        feature_group="parent_ml01_approved",
        leakage_risk="inherited_from_ml01_approved_contract",
        selection_note="carried forward from ML-01 approved ML input contract",
    )

# ML-01 승인본에 이미 materialized된 categorical feature는 중복 encode하지 않는다.
TIME_NATIVE_CATEGORICAL_ROWS = []
ML00_NATIVE_CATEGORICAL_ROWS = []

# 최종 contract row 리스트.
# 순서가 의미를 가진다: source parquet 컬럼 순서 → ML-02 Stage 1 신규 수치 row.
# 다운스트림(write/validate)에서 이 순서를 그대로 사용한다.
USER_CONTRACT_ROWS = [
    # 1) source parquet 전체 컬럼을 원본 순서대로 contract에 명시.
    #    - SOURCE_CONTRACT_OVERRIDES에 있으면 ML 입력용 row로 override.
    #    - 아니면 inventory row(used_in_ml=FALSE)로 보존.
    *(
        SOURCE_CONTRACT_OVERRIDES[column]
        if column in SOURCE_CONTRACT_OVERRIDES
        else source_inventory_contract_row(column)
        for column in source_columns
    ),

    # 2) ML-01 승인본에 이미 존재하는 categorical feature는 중복 encode하지 않는다.
    *TIME_NATIVE_CATEGORICAL_ROWS,

    # 3) ML-00/ML-01 categorical feature는 parent 승인 contract 기준 carry_forward로 승계한다.
    *ML00_NATIVE_CATEGORICAL_ROWS,

    # 4) Stage 1 accountstats 후보 전체를 수치형(q) feature로 build.
    #    STAGE1_FEATURE_SPECS는 ml02_stage1_feature_specs(used_in_ml=True)에서 생성된 spec 목록.
    *[stage1_contract_row(spec.output_col) for spec in STAGE1_FEATURE_SPECS],
]

# parent에서 승인된 materialized categorical feature는 carry_forward하되, contract 검증용 category 값을 parent manifest에서 승계한다.
parent_category_values = parent_encoding_manifest.get("category_values", {})
CATEGORY_VALUES = {
    column: parent_category_values[column]
    for column in parent_selected_df.loc[
        parent_selected_df["xgb_feature_type"].astype(str).str.strip() == "c",
        "column_name",
    ].astype(str).str.strip()
    if column in parent_category_values
}

# 구성 결과 sanity check.
# 기대값: user_contract_row_count == len(source_columns) + len(STAGE1_FEATURE_SPECS)
print("source_contract_row_count:",         len(source_columns))
print("source_override_row_count:",         len(SOURCE_CONTRACT_OVERRIDES))
print("time_native_categorical_row_count:", len(TIME_NATIVE_CATEGORICAL_ROWS))
print("ml00_native_categorical_row_count:", len(ML00_NATIVE_CATEGORICAL_ROWS))
print("stage1_contract_row_count:",         len(STAGE1_FEATURE_SPECS))
print("user_contract_row_count:",           len(USER_CONTRACT_ROWS))
print("category_values_columns:",           sorted(CATEGORY_VALUES))

source_contract_row_count: 108
source_override_row_count: 76
time_native_categorical_row_count: 0
ml00_native_categorical_row_count: 0
stage1_contract_row_count: 28
user_contract_row_count: 136
category_values_columns: ['cat__currency_pair__xgb_cat', 'cat__payment_currency__xgb_cat', 'cat__payment_format__xgb_cat', 'cat__receiving_currency__xgb_cat', 'cat__time_dayofweek__xgb_cat', 'cat__time_is_weekend__xgb_cat']


# 05_최종 contract 저장 및 검증

In [19]:
# ============================================================
# 5.0 최종 contract 저장 및 검증
# ============================================================
# 이 셀의 역할:
# 4.1에서 만든 USER_CONTRACT_ROWS를 실제 산출물(fb_input_feature_contract.csv,
# fb_input_category_values.json)로 디스크에 기록하고, 기록된 contract가 downstream
# feature_build 단계에서 그대로 사용 가능한지 형식·정합성을 검증한다.
#
# 산출물
# - FB_INPUT_CONTRACT_PATH       : feature build의 입력 contract (CSV, UTF-8 BOM)
# - FB_INPUT_CATEGORY_VALUES_PATH: 수동 지정 categorical 값 사전 (JSON)
#
# 저장 전 사전 검증 (모두 실패 시 즉시 raise — 깨진 contract를 디스크에 남기지 않음)
# - USER_CONTRACT_ROWS 비어있는지
# - CONTRACT_COLUMNS schema 충족 (누락 컬럼은 빈 문자열로 채움)
# - 컬럼 순서를 CONTRACT_COLUMNS 기준으로 고정
# - column_name 중복 여부
# - 수동(non-encode) categorical 컬럼과 CATEGORY_VALUES 사전 정의 일치 여부
# - 기존 산출물 파일 overwrite 정책(OVERWRITE_CONTRACT_OUTPUTS)
#
# 저장 후 사후 검증
# - validate_fb_input_contract()로 contract 문법·source parquet 호환성 확인
#
# 실패 시 조치
# - row가 없는 경우: 4.1 셀에서 USER_CONTRACT_ROWS를 먼저 작성
# - 중복/누락 카테고리: 4.1 셀의 helper 호출을 점검

# --- 1) 입력 존재 검증 ---------------------------------------------------
# contract row가 하나도 없으면 저장할 contract가 없으므로 즉시 중단한다.
if not USER_CONTRACT_ROWS:
    raise ValueError("USER_CONTRACT_ROWS is empty. 데이터/후보를 확인한 뒤 contract row를 직접 작성해야 합니다.")


# --- 2) DataFrame 변환 및 schema 정규화 ----------------------------------
# 사용자가 작성한 row list(dict들)를 DataFrame으로 변환한다.
# 각 row는 carry_forward_contract_row(), categorical_contract_row(),
# encode_contract_row(), stage1_contract_row(), source_inventory_contract_row() 중
# 어느 하나가 만든 dict다.
contract_df = pd.DataFrame(USER_CONTRACT_ROWS)

# CONTRACT_COLUMNS에 정의된 전체 contract schema를 강제한다.
# 일부 helper가 특정 컬럼을 채우지 않았더라도 누락 컬럼은 빈 문자열로 채워
# downstream 파서에서 KeyError가 나지 않도록 한다.
for column in CONTRACT_COLUMNS:
    if column not in contract_df.columns:
        contract_df[column] = ""

# 컬럼 순서를 CONTRACT_COLUMNS 기준으로 고정한다.
# 저장 CSV의 schema(컬럼 개수·순서)가 run마다 항상 동일해야
# 외부 도구(diff/스키마 비교)가 안정적으로 작동한다.
contract_df = contract_df[CONTRACT_COLUMNS].copy()


# --- 3) column_name 중복 검사 -------------------------------------------
# 같은 column_name이 두 번 들어가면 어떤 feature row를 사용할지 모호해진다.
# 예: 같은 feature를 stage1_contract_row와 carry_forward_contract_row로 동시에 넣은 경우.
# 앞뒤 공백 차이도 같은 이름으로 보기 위해 strip 후 비교한다.
if contract_df["column_name"].astype(str).str.strip().duplicated().any():
    duplicated = contract_df.loc[
        contract_df["column_name"].astype(str).str.strip().duplicated(),
        "column_name",
    ].tolist()
    # 너무 길어지지 않도록 앞 30개만 노출한다.
    raise ValueError(f"duplicated contract column_name values: {duplicated[:30]}")


# --- 4) categorical 컬럼 추출 -------------------------------------------
# (a) ML 입력으로 사용되는 모든 categorical 컬럼 = used_in_ml == "TRUE" AND xgb_feature_type == "c"
#     디버깅·로깅 용도로 미리 뽑아둔다 (현재 셀 안에서 직접 검증에 쓰이지는 않음).
categorical_columns = contract_df.loc[
    (contract_df["used_in_ml"].astype(str).str.strip() == "TRUE")
    & (contract_df["xgb_feature_type"].astype(str).str.strip() == "c"),
    "column_name",
].astype(str).str.strip().tolist()

# (b) 그 중에서 build_action != "encode" 인 것 = 수동 카테고리 컬럼.
#     encode는 build 단계에서 값 집합을 자동 추론하지만, 수동 카테고리는
#     CATEGORY_VALUES로 미리 허용 값 집합을 명시해야 한다.
manual_categorical_columns = contract_df.loc[
    (contract_df["used_in_ml"].astype(str).str.strip() == "TRUE")
    & (contract_df["xgb_feature_type"].astype(str).str.strip() == "c")
    & (contract_df["build_action"].astype(str).str.strip() != "encode"),
    "column_name",
].astype(str).str.strip().tolist()


# --- 5) 수동 카테고리 ↔ CATEGORY_VALUES 정합성 검증 -----------------------
# CATEGORY_VALUES에 빠진 manual categorical이 있으면 build 단계에서 미정의 값으로 실패한다.
# 반대로 CATEGORY_VALUES에만 있고 실제 contract에는 없는 컬럼이 있으면 죽은 설정이다.
missing_category_values = sorted(set(manual_categorical_columns) - set(CATEGORY_VALUES))
extra_category_values   = sorted(set(CATEGORY_VALUES) - set(manual_categorical_columns))

if missing_category_values:
    raise ValueError(f"CATEGORY_VALUES is missing categorical columns: {missing_category_values}")

if extra_category_values:
    raise ValueError(f"CATEGORY_VALUES contains unused columns: {extra_category_values}")

# 출력 JSON에 쓸 dict를 만든다.
# 값 비교를 안정시키기 위해 모두 문자열로 캐스팅한다(예: int 12 vs "12" 혼용 방지).
category_values_for_output = {
    column: [str(value) for value in CATEGORY_VALUES[column]]
    for column in manual_categorical_columns
}


# --- 6) overwrite 정책 점검 ---------------------------------------------
# 기존 산출물이 이미 있고 OVERWRITE_CONTRACT_OUTPUTS=False이면 덮어쓰지 않고 중단한다.
# 의도치 않은 contract 교체로 인한 재현성 손상 방지.
if FB_INPUT_CONTRACT_PATH.exists() and not OVERWRITE_CONTRACT_OUTPUTS:
    raise FileExistsError(
        "contract already exists. "
        f"Set OVERWRITE_CONTRACT_OUTPUTS=True to replace it: {FB_INPUT_CONTRACT_PATH}"
    )
if FB_INPUT_CATEGORY_VALUES_PATH.exists() and not OVERWRITE_CONTRACT_OUTPUTS:
    raise FileExistsError(
        "category values already exist. "
        f"Set OVERWRITE_CONTRACT_OUTPUTS=True to replace it: {FB_INPUT_CATEGORY_VALUES_PATH}"
    )


# --- 7) 산출물 저장 ------------------------------------------------------
# 저장 경로의 parent directory가 없으면 생성한다. 예: ml/ml-02/fb_inputs/r00/
FB_INPUT_CONTRACT_PATH.parent.mkdir(parents=True, exist_ok=True)

# 최종 contract CSV 저장.
# - index=False: 행 인덱스 컬럼을 CSV에 쓰지 않는다.
# - encoding="utf-8-sig": Excel에서 한글이 깨지지 않도록 BOM 포함 UTF-8로 저장.
contract_df.to_csv(FB_INPUT_CONTRACT_PATH, index=False, encoding="utf-8-sig")

# 수동 카테고리 사전을 JSON으로 저장한다.
# - artifact_prefix: 이 산출물이 속한 run prefix (downstream에서 contract와 짝 맞춤 검증).
# - fit_split="manual": 카테고리 값을 데이터에서 학습한 게 아니라 사용자가 지정했다는 표식.
# - source_contract_path: 이 카테고리 사전과 짝을 이루는 contract CSV의 경로.
# - category_values: 컬럼별 허용 카테고리 값 리스트(모두 문자열).
category_payload = {
    "artifact_prefix": ARTIFACT_PREFIX,
    "fit_split": "manual",
    "source_contract_path": str(FB_INPUT_CONTRACT_PATH),
    "category_values": category_values_for_output,
}
FB_INPUT_CATEGORY_VALUES_PATH.write_text(
    json.dumps(category_payload, indent=2, ensure_ascii=False),  # 한글/특수문자 보존
    encoding="utf-8",
)


# --- 8) 저장 후 검증 -----------------------------------------------------
# 디스크에 기록된 contract를 다시 읽어 형식과 소스 호환성을 점검한다.
# 검증 항목(예시):
# - 필수 컬럼 존재 여부
# - used_in_ml / build_in_fb / materialized 값의 허용 형식 (TRUE/FALSE 등)
# - artifact_prefix가 현재 run의 ARTIFACT_PREFIX와 일치
# - carry_forward로 선택된 source_column이 source parquet에 실제 존재
# - 금지된(예약된) feature 이름이 사용되지 않았는지
validation = validate_fb_input_contract(
    FB_INPUT_CONTRACT_PATH,
    source_data_path=SOURCE_PARQUET_PATH,
    artifact_prefix=ARTIFACT_PREFIX,
)


# --- 9) 결과 요약 출력 ---------------------------------------------------
# total_rows   : contract에 기록된 전체 row 수 (inventory 포함)
# selected_count: used_in_ml=TRUE인 row 수 (실제 ML 입력 feature 수)
print("contract saved", FB_INPUT_CONTRACT_PATH)
print("category values saved", FB_INPUT_CATEGORY_VALUES_PATH)
print("total_rows", validation.total_rows)
print("selected_count", validation.selected_count)

# 최종 contract 내용을 노트북에서 표로 확인한다.
# 시각적으로 used_in_ml/encoding/feature_group 등을 한 번 더 검토하기 위함이다.
display(contract_df)

contract saved /mnt/d/AML_git/Graph_AML/ml/ml-02/fb_inputs/r01/ml_02__r01_fb_input_feature_contract.csv
category values saved /mnt/d/AML_git/Graph_AML/ml/ml-02/fb_inputs/r01/ml_02__r01_category_values.json
total_rows 136
selected_count 101


,contract_version,artifact_prefix,column_name,used_in_ml,source_column,column_origin,encoding,build_action,build_in_fb,xgb_feature_type,...,source_data_path,feature_spec_name,encoding_params,materialized,observed_dtype,missing_count,missing_rate,unknown_category_count,fit_split,note
0,1,ml_02__r01,tx_id,FALSE,tx_id,preprocessing,passthrough,carry_forward,FALSE,,...,/mnt/d/AML_git/Graph_AML/ml/ml-01/ml_inputs/r02/ml_01__r02_Xy_all.parquet,,,TRUE,string,,,,,source parquet column preserved; not selected for ML by default
1,1,ml_02__r01,timestamp,FALSE,timestamp,preprocessing,passthrough,carry_forward,FALSE,,...,/mnt/d/AML_git/Graph_AML/ml/ml-01/ml_inputs/r02/ml_01__r02_Xy_all.parquet,,,TRUE,timestamp[us],,,,,source parquet column preserved; not selected for ML by default
2,1,ml_02__r01,split,FALSE,split,preprocessing,passthrough,carry_forward,FALSE,,...,/mnt/d/AML_git/Graph_AML/ml/ml-01/ml_inputs/r02/ml_01__r02_Xy_all.parquet,,,TRUE,string,,,,,source parquet column preserved; not selected for ML by default
3,1,ml_02__r01,label,FALSE,label,preprocessing,passthrough,carry_forward,FALSE,,...,/mnt/d/AML_git/Graph_AML/ml/ml-01/ml_inputs/r02/ml_01__r02_Xy_all.parquet,,,TRUE,int8,,,,,source parquet column preserved; not selected for ML by default
4,1,ml_02__r01,sender_account,FALSE,sender_account,preprocessing,passthrough,carry_forward,FALSE,,...,/mnt/d/AML_git/Graph_AML/ml/ml-01/ml_inputs/r02/ml_01__r02_Xy_all.parquet,,,TRUE,string,,,,,source parquet column preserved; not selected for ML by default
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
131,1,ml_02__r01,accountstats__receiver__in__amount__sum__w7d,TRUE,accountstats__receiver__in__amount__sum__w7d,current_build,passthrough,build,TRUE,q,...,/mnt/d/AML_git/Graph_AML/ml/ml-01/ml_inputs/r02/ml_01__r02_Xy_all.parquet,rolling_entity_agg,"{""input_cols"": {""current_entity_col"": ""receiver_account_id"", ""history_entity_col"": ""receiver_account_id"", ""timestamp_col"": ""timestamp"", ""value_col"": ""amount_received""}, ""params"": {""agg"": ""sum"", ""closed"": ""left"", ""fill_value"": 0.0, ""window"": ""7d""}}",FALSE,,,,,,to be built by 01_ml_02_fb_build_features.ipynb
132,1,ml_02__r01,accountstats__receiver__out__amount__sum__w6h,TRUE,accountstats__receiver__out__amount__sum__w6h,current_build,passthrough,build,TRUE,q,...,/mnt/d/AML_git/Graph_AML/ml/ml-01/ml_inputs/r02/ml_01__r02_Xy_all.parquet,rolling_entity_agg,"{""input_cols"": {""current_entity_col"": ""receiver_account_id"", ""history_entity_col"": ""sender_account_id"", ""timestamp_col"": ""timestamp"", ""value_col"": ""amount""}, ""params"": {""agg"": ""sum"", ""closed"": ""left"", ""fill_value"": 0.0, ""window"": ""6h""}}",FALSE,,,,,,to be built by 01_ml_02_fb_build_features.ipynb
133,1,ml_02__r01,accountstats__receiver__out__amount__sum__w1d,TRUE,accountstats__receiver__out__amount__sum__w1d,current_build,passthrough,build,TRUE,q,...,/mnt/d/AML_git/Graph_AML/ml/ml-01/ml_inputs/r02/ml_01__r02_Xy_all.parquet,rolling_entity_agg,"{""input_cols"": {""current_entity_col"": ""receiver_account_id"", ""history_entity_col"": ""sender_account_id"", ""timestamp_col"": ""timestamp"", ""value_col"": ""amount""}, ""params"": {""agg"": ""sum"", ""closed"": ""left"", ""fill_value"": 0.0, ""window"": ""1d""}}",FALSE,,,,,,to be built by 01_ml_02_fb_build_features.ipynb
134,1,ml_02__r01,accountstats__receiver__out__amount__sum__w3d,TRUE,accountstats__receiver__out__amount__sum__w3d,current_build,passthrough,build,TRUE,q,...,/mnt/d/AML_git/Graph_AML/ml/ml-01/ml_inputs/r02/ml_01__r02_Xy_all.parquet,rolling_entity_agg,"{""input_cols"": {""current_entity_col"": ""receiver_account_id"", ""history_entity_col"": ""sender_account_id"", ""timestamp_col"": ""timestamp"", ""value_col"": ""amount""}, ""params"": {""agg"": ""sum"", ""closed"": ""left"", ""fill_value"": 0.0, ""window"": ""3d""}}",FALSE,,,,,,to be built by 01_ml_02_fb_build_features.ipynb
